# Motion

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")
print(f"Python version: {sys.version}")

In [ ]:
from femtum_sdk.api.contracts.v1.hardware.health_pb2 import HealthRequest
from femtum_sdk.api import HardwareApi

api_host = "10.20.200.10:5210"

with HardwareApi(host=api_host) as hardware_api:
    print(hardware_api.health.GetHealth(HealthRequest()))


## Read Current Position

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasureUnit
from femtum_sdk.api.contracts.v1.dto.motion_pb2 import PositionMeasure3D
from femtum_sdk.api.contracts.v1.hardware.motion_pb2 import ReadWaferStageCurrentPositionRequest, ReadWaferStageCurrentPositionResponse

def print_position(position: PositionMeasure3D):
    print(f"X = {position.X.Value} {DistanceMeasureUnit.Name(position.X.Unit)}")
    print(f"Y = {position.Y.Value} {DistanceMeasureUnit.Name(position.Y.Unit)}")
    print(f"Z = {position.Z.Value} {DistanceMeasureUnit.Name(position.Z.Unit)}")

with HardwareApi(host=api_host) as hardware_api:
    response: ReadWaferStageCurrentPositionResponse = hardware_api.wafer_stage_motion.ReadCurrentPosition(ReadWaferStageCurrentPositionRequest())
    position = response.Position

    print_position(position)


## Relative Move

⚠️ **Warning:** Before running a relative move, ensure the path is clear to prevent collisions or equipment damage.

### All At Once

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasure
from femtum_sdk.api.contracts.v1.hardware.motion_pb2 import AllAtOnceMove, MoveOrder, MoveToWaferStageRelativePositionRequest, MoveToWaferStageRelativePositionResponse


with HardwareApi(host=api_host) as hardware_api:
    current_position: PositionMeasure3D = hardware_api.wafer_stage_motion.ReadCurrentPosition(ReadWaferStageCurrentPositionRequest()).Position

    response: MoveToWaferStageRelativePositionResponse = hardware_api.wafer_stage_motion.MoveToRelativePosition(MoveToWaferStageRelativePositionRequest(
        X=DistanceMeasure(
            Value=100,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=100, 
            Unit=DistanceMeasureUnit.Microns
        ),
        Order=MoveOrder(AllAtOnce=AllAtOnceMove())
    ))

    response_after_move_position: PositionMeasure3D = response.Position
    read_after_move_position: PositionMeasure3D = hardware_api.wafer_stage_motion.ReadCurrentPosition(ReadWaferStageCurrentPositionRequest()).Position

    print("Before move:")
    print_position(current_position)
    print("After move:")
    print_position(read_after_move_position)
    tolerance = 0.5  # Microns; the stage won't land on the exact target
    assert abs(read_after_move_position.X.Value - (current_position.X.Value + 100)) <= tolerance
    assert abs(read_after_move_position.Y.Value - (current_position.Y.Value + 100)) <= tolerance
    assert abs(read_after_move_position.Z.Value - current_position.Z.Value) <= tolerance

    assert abs(response_after_move_position.X.Value - (current_position.X.Value + 100)) <= tolerance
    assert abs(response_after_move_position.Y.Value - (current_position.Y.Value + 100)) <= tolerance
    assert abs(response_after_move_position.Z.Value - current_position.Z.Value) <= tolerance


### Ordered

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasure
from femtum_sdk.api.contracts.v1.dto.motion_pb2 import Axis
from femtum_sdk.api.contracts.v1.hardware.motion_pb2 import MoveOrder, MoveToWaferStageRelativePositionRequest, MoveToWaferStageRelativePositionResponse, OrderedMove


with HardwareApi(host=api_host) as hardware_api:
    current_position: PositionMeasure3D = hardware_api.wafer_stage_motion.ReadCurrentPosition(ReadWaferStageCurrentPositionRequest()).Position

    move_a: MoveToWaferStageRelativePositionResponse = hardware_api.wafer_stage_motion.MoveToRelativePosition(MoveToWaferStageRelativePositionRequest(
        X=DistanceMeasure(
            Value=10000,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=10000, 
            Unit=DistanceMeasureUnit.Microns
        ),
        Z=DistanceMeasure(
            Value=10000,
            Unit=DistanceMeasureUnit.Microns
        ),
        Order=MoveOrder(Ordered=OrderedMove(Axis=[Axis.AXIS_Z, Axis.AXIS_Y, Axis.AXIS_X])) # Move Z first, then Y, then X
    ))
    print("Move A:")
    print_position(move_a.Position)

    import time
    time.sleep(2.5) # Wait for human to see the move

    move_b: MoveToWaferStageRelativePositionResponse = hardware_api.wafer_stage_motion.MoveToRelativePosition(MoveToWaferStageRelativePositionRequest(
        X=DistanceMeasure(
            Value=-10000,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=-10000, 
            Unit=DistanceMeasureUnit.Microns
        ),
        Z=DistanceMeasure(
            Value=-10000,
            Unit=DistanceMeasureUnit.Microns
        ),
        Order=MoveOrder(Ordered=OrderedMove(Axis=[Axis.AXIS_X, Axis.AXIS_Y, Axis.AXIS_Z])) # Move X first, then Y, then Z
    ))

    print("Move B:")
    print_position(move_b.Position)

    
    tolerance = 0.5  # Microns; the stage won't land on the exact target
    assert abs(read_after_move_position.X.Value - (move_b.Position.X.Value)) <= tolerance
    assert abs(read_after_move_position.Y.Value - (move_b.Position.Y.Value)) <= tolerance
    assert abs(read_after_move_position.Z.Value - (move_b.Position.Z.Value)) <= tolerance


## Absolute Move

⚠️ **Warning:** Before running a relative move, ensure the path is clear to prevent collisions or equipment damage.

### All at Once

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasure
from femtum_sdk.api.contracts.v1.hardware.motion_pb2 import MoveToWaferStageAbsolutePositionRequest, MoveToWaferStageAbsolutePositionResponse


with HardwareApi(host=api_host) as hardware_api:
    move_a_response: MoveToWaferStageAbsolutePositionResponse = hardware_api.wafer_stage_motion.MoveToAbsolutePosition(MoveToWaferStageAbsolutePositionRequest(
        X=DistanceMeasure(
            Value=25200.0,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=6574.0, 
            Unit=DistanceMeasureUnit.Microns
        ),
    ))
    move_a_position: PositionMeasure3D = move_a_response.Position

    move_b_response: MoveToWaferStageAbsolutePositionResponse = hardware_api.wafer_stage_motion.MoveToAbsolutePosition(MoveToWaferStageAbsolutePositionRequest(
        X=DistanceMeasure(
            Value=25300.0,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=6580.0, 
            Unit=DistanceMeasureUnit.Microns
        ),
    ))
    move_b_position: PositionMeasure3D = move_b_response.Position
    

    read_after_move_position: PositionMeasure3D = hardware_api.wafer_stage_motion.ReadCurrentPosition(ReadWaferStageCurrentPositionRequest()).Position

    print("Before move:")
    print_position(current_position)
    print("After move:")
    print_position(read_after_move_position)
    tolerance = 0.5  # Microns; the stage won't land on the exact target
    assert abs(read_after_move_position.X.Value - (move_b_position.X.Value)) <= tolerance
    assert abs(read_after_move_position.Y.Value - (move_b_position.Y.Value)) <= tolerance





### Ordered

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasure
from femtum_sdk.api.contracts.v1.hardware.motion_pb2 import MoveToWaferStageAbsolutePositionRequest, MoveToWaferStageAbsolutePositionResponse


with HardwareApi(host=api_host) as hardware_api:
    move_a_response: MoveToWaferStageAbsolutePositionResponse = hardware_api.wafer_stage_motion.MoveToAbsolutePosition(MoveToWaferStageAbsolutePositionRequest(
        X=DistanceMeasure(
            Value=35200.0,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=8574.0, 
            Unit=DistanceMeasureUnit.Microns
        ),
        Order=MoveOrder(Ordered=OrderedMove(Axis=[Axis.AXIS_Y, Axis.AXIS_X]))
    ))
    move_a_position: PositionMeasure3D = move_a_response.Position

    import time
    time.sleep(2.5) # Wait for human to see the move

    move_b_response: MoveToWaferStageAbsolutePositionResponse = hardware_api.wafer_stage_motion.MoveToAbsolutePosition(MoveToWaferStageAbsolutePositionRequest(
        X=DistanceMeasure(
            Value=25300.0,
            Unit=DistanceMeasureUnit.Microns
        ),
        Y=DistanceMeasure(
            Value=6580.0, 
            Unit=DistanceMeasureUnit.Microns
        ),
        Order=MoveOrder(Ordered=OrderedMove(Axis=[Axis.AXIS_X, Axis.AXIS_Y]))
    ))
    move_b_position: PositionMeasure3D = move_b_response.Position
    
    print("Before move:")
    print_position(current_position)
    print("After move:")
    print_position(read_after_move_position)

